In [33]:
import pandas as pd

all_2247 = pd.read_csv("../Data/ST-2247.csv")
all_2252 = pd.read_csv("../Data/ST-2252.csv")
all_2425 = pd.read_csv("../Data/ST-2425.csv")

In [ ]:
# 필요없는 컬럼 제거

def drop_unnamed_columns(df):
    cols_to_drop = [c for c in df.columns if "Unnamed" in str(c)]
    return df.drop(columns=cols_to_drop, inplace=True)


for df in [all_2247, all_2252, all_2425]:
    drop_unnamed_columns(df)

In [35]:
all_2247

,기준_날짜,시작_대여소_ID,시작_대여소명,종료_대여소_ID,종료_대여소명,전체_건수,전체_이용_분,전체_이용_거리,대여소_ID,주소1,...,경도,이용기준,이용시각,요일,주말,날씨시각,기온,강수량,적설량,풍속
0,20240101,ST-1036,역촌동_001_1,ST-2247,신사2동_013_1,1.0,74.0,740.0,ST-1036,서울특별시 은평구 연서로 124,...,126.916794,도착시간,2024-01-01 16:40:00,월,주말,2024-01-01 16:00:00,5.9,0.0,0.0,10.8
1,20240101,ST-463,증산동_004_1,ST-2247,신사2동_013_1,1.0,8.0,1268.0,ST-463,서울특별시 은평구 증산동 199-8,...,126.909897,도착시간,2024-01-01 19:13:00,월,주말,2024-01-01 19:00:00,0.2,0.0,0.0,8.0
2,20240101,ST-2247,신사2동_013_1,ST-2247,신사2동_013_1,1.0,11.0,1196.0,ST-2247,서울특별시 은평구 신사동 178-9,...,126.906082,도착시간,2024-01-01 18:16:00,월,주말,2024-01-01 18:00:00,2.0,0.0,0.0,7.6
3,20240101,ST-455,응암1동_032_1,ST-2247,신사2동_013_1,2.0,44.0,4070.0,ST-455,서울특별시 은평구 은평로 85 CJ드림시티,...,126.916985,도착시간,2024-01-01 22:30:00,월,주말,2024-01-01 22:00:00,-0.1,0.0,0.0,6.2
4,20240101,ST-455,응암1동_032_1,ST-2247,신사2동_013_1,2.0,44.0,4070.0,ST-455,서울특별시 은평구 은평로 85 CJ드림시티,...,126.916985,도착시간,2024-01-01 22:49:00,월,주말,2024-01-01 22:00:00,-0.1,0.0,0.0,6.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25708,20241231,ST-463,증산동_011_1,ST-2247,신사2동_013_1,1.0,8.0,1144.0,ST-463,서울특별시 은평구 증산동 199-8,...,126.909897,도착시간,2024-12-31 14:33:00,화,주중,2024-12-31 14:00:00,1.8,0.0,0.0,12.5
25709,20241231,ST-2247,신사2동_013_1,ST-2537,상암동_029_1,1.0,15.0,2141.0,ST-2247,서울특별시 은평구 신사동 178-9,...,126.906082,출발시간,2024-12-31 15:30:00,화,주중,2024-12-31 15:00:00,1.8,0.0,0.0,12.4
25710,20241231,ST-2247,신사2동_013_1,ST-2240,이촌2동_012_1,1.0,65.0,13623.0,ST-2247,서울특별시 은평구 신사동 178-9,...,126.906082,출발시간,2024-12-31 10:00:00,화,주중,2024-12-31 10:00:00,-0.7,0.0,0.0,12.2
25711,20241231,ST-2825,응암2동_050_1,ST-2247,신사2동_013_1,1.0,12.0,1545.0,ST-2825,서울특별시 은평구 은평구 응암로 278 맞은편(응암동 767-1 공원 앞),...,126.919624,도착시간,2024-12-31 07:52:00,화,주중,2024-12-31 07:00:00,-3.5,0.0,0.0,13.0


In [ ]:
# 상대 대수 구하기
import pandas as pd

def build_relative_count_df(
    df,
    time_col="날씨시각",
    usage_col="이용기준",
    count_col="전체_건수",
    arrival_value="도착시간",
    depart_value="출발시간",
    result_col="상대대수",
):
    # 상대대수 계산
    signed = df[count_col].where(df[usage_col] == arrival_value,
                                 -df[count_col])
    df[result_col] = signed

    # 유지할 컬럼
    meta_cols = ["주소1", "요일", "주말", "기온", "강수량", "적설량", "풍속"]

    # 그룹바이: 상대대수는 합, 나머지는 대표값
    agg_map = {result_col: "sum"}
    for c in meta_cols:
        agg_map[c] = "first"

    out = (
        df.groupby(time_col, as_index=False)
          .agg(agg_map)
          .sort_values(time_col)
          .reset_index(drop=True)
    )

    return out


all_2247 = build_relative_count_df(all_2247)
all_2252 = build_relative_count_df(all_2252)
all_2425 = build_relative_count_df(all_2425)


In [37]:
all_2247

,날씨시각,상대대수,주소1,요일,주말,기온,강수량,적설량,풍속
0,2024-01-01 00:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,주말,-1.9,0.0,0.0,6.3
1,2024-01-01 04:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,주말,-2.8,0.0,0.0,7.6
2,2024-01-01 14:00:00,2.0,서울특별시 은평구 응암동 604-5,월,주말,6.4,0.0,0.0,9.4
3,2024-01-01 16:00:00,4.0,서울특별시 은평구 연서로 124,월,주말,5.9,0.0,0.0,10.8
4,2024-01-01 18:00:00,2.0,서울특별시 은평구 신사동 178-9,월,주말,2.0,0.0,0.0,7.6
...,...,...,...,...,...,...,...,...,...
5399,2024-12-31 18:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,주중,-1.2,0.0,0.0,6.5
5400,2024-12-31 19:00:00,2.0,서울특별시 은평구 신사동 178-9,화,주중,-1.9,0.0,0.0,6.3
5401,2024-12-31 20:00:00,2.0,서울특별시 은평구 신사동 352-2,화,주중,-2.5,0.0,0.0,4.7
5402,2024-12-31 21:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,주중,-2.9,0.0,0.0,4.1


In [ ]:
# 현재 대수 구하기
import pandas as pd

def add_current_count_with_reset(
    df,
    time_col="날씨시각",
    relative_col="상대대수",
    current_col="현재대수",
    reset_hours=(4, 16),
):
    out = df.sort_values(time_col).reset_index(drop=True).copy()

    dt = pd.to_datetime(out[time_col])
    hour = dt.dt.hour

    # 04:00~15:59는 shift=0, 16:00~03:59는 shift=1로 묶기
    shift = (hour >= reset_hours[1]).astype(int)

    # 04:00 기준으로 날짜를 맞추기 위해 4시간 빼고 날짜 그룹 생성
    day_bucket = (dt - pd.Timedelta(hours=reset_hours[0])).dt.date

    group_key = day_bucket.astype(str) + "_" + shift.astype(str)

    out[current_col] = out.groupby(group_key)[relative_col].cumsum()
    return out

all_2247 = add_current_count_with_reset(all_2247)
all_2252 = add_current_count_with_reset(all_2252)
all_2425 = add_current_count_with_reset(all_2425)

In [39]:
all_2247

,날씨시각,상대대수,주소1,요일,주말,기온,강수량,적설량,풍속,현재대수
0,2024-01-01 00:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,주말,-1.9,0.0,0.0,6.3,-2.0
1,2024-01-01 04:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,주말,-2.8,0.0,0.0,7.6,-2.0
2,2024-01-01 14:00:00,2.0,서울특별시 은평구 응암동 604-5,월,주말,6.4,0.0,0.0,9.4,0.0
3,2024-01-01 16:00:00,4.0,서울특별시 은평구 연서로 124,월,주말,5.9,0.0,0.0,10.8,4.0
4,2024-01-01 18:00:00,2.0,서울특별시 은평구 신사동 178-9,월,주말,2.0,0.0,0.0,7.6,6.0
...,...,...,...,...,...,...,...,...,...,...
5399,2024-12-31 18:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,주중,-1.2,0.0,0.0,6.5,-4.0
5400,2024-12-31 19:00:00,2.0,서울특별시 은평구 신사동 178-9,화,주중,-1.9,0.0,0.0,6.3,-2.0
5401,2024-12-31 20:00:00,2.0,서울특별시 은평구 신사동 352-2,화,주중,-2.5,0.0,0.0,4.7,0.0
5402,2024-12-31 21:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,주중,-2.9,0.0,0.0,4.1,-2.0


In [40]:
all_2247.to_csv("../Data/ST-2247_current.csv", index=False)
all_2252.to_csv("../Data/ST-2252_current.csv", index=False)
all_2425.to_csv("../Data/ST-2425_current.csv", index=False)

--------
#### 전처리

In [ ]:
# 주말 인덱싱
def holiday_indexing(df):
    df["주말"] = df["주말"].map({"주말": 1, "주중": 0}).fillna(df["주말"]).astype(int)
    return df

all_2247 = holiday_indexing(all_2247)
all_2252 = holiday_indexing(all_2252)
all_2425 = holiday_indexing(all_2425)

In [44]:
all_2247

,날씨시각,상대대수,주소1,요일,주말,기온,강수량,적설량,풍속,현재대수
0,2024-01-01 00:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,1,-1.9,0.0,0.0,6.3,-2.0
1,2024-01-01 04:00:00,-2.0,서울특별시 은평구 신사동 178-9,월,1,-2.8,0.0,0.0,7.6,-2.0
2,2024-01-01 14:00:00,2.0,서울특별시 은평구 응암동 604-5,월,1,6.4,0.0,0.0,9.4,0.0
3,2024-01-01 16:00:00,4.0,서울특별시 은평구 연서로 124,월,1,5.9,0.0,0.0,10.8,4.0
4,2024-01-01 18:00:00,2.0,서울특별시 은평구 신사동 178-9,월,1,2.0,0.0,0.0,7.6,6.0
...,...,...,...,...,...,...,...,...,...,...
5399,2024-12-31 18:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,0,-1.2,0.0,0.0,6.5,-4.0
5400,2024-12-31 19:00:00,2.0,서울특별시 은평구 신사동 178-9,화,0,-1.9,0.0,0.0,6.3,-2.0
5401,2024-12-31 20:00:00,2.0,서울특별시 은평구 신사동 352-2,화,0,-2.5,0.0,0.0,4.7,0.0
5402,2024-12-31 21:00:00,-2.0,서울특별시 은평구 신사동 178-9,화,0,-2.9,0.0,0.0,4.1,-2.0


------
#### 머신러닝

In [ ]:
# !pip install scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 878.0 kB/s  0:00:08 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 614.8 kB/s  0:00:32m0:00:0100:02
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [45]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def train_rf_with_search_time_split(
    df,
    time_col="날씨시각",
    feature_cols=("주말", "기온", "강수량", "적설량", "풍속"),
    target_col="현재대수",
    test_size=0.2,
    val_size=0.2,
    random_state=42,
):
    # 1) 시간순 정렬
    df_sorted = df.sort_values(time_col).reset_index(drop=True)

    X = df_sorted[list(feature_cols)]
    y = df_sorted[target_col]

    n = len(df_sorted)
    n_test = int(n * test_size)
    n_val = int(n * val_size)

    n_train = n - n_val - n_test
    if n_train <= 0:
        raise ValueError("train/val/test 비율이 너무 큽니다.")

    # 2) 시간순 분할
    X_train, y_train = X.iloc[:n_train], y.iloc[:n_train]
    X_val, y_val = X.iloc[n_train:n_train+n_val], y.iloc[n_train:n_train+n_val]
    X_test, y_test = X.iloc[n_train+n_val:], y.iloc[n_train+n_val:]

    # 3) 파이프라인 (정규화 + 랜덤포레스트)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=random_state, n_jobs=-1))
    ])

    # 4) 하이퍼파라미터 탐색
    param_grid = {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 8, 12],
        "model__min_samples_split": [2, 5],
        "model__min_samples_leaf": [1, 2],
        "model__max_features": ["sqrt", "log2"]
    }

    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        cv=3,
        scoring="r2",
        n_jobs=-1
    )
    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    # 5) 점수 출력
    def _eval(name, X_split, y_split):
        pred = best_model.predict(X_split)
        return {
            "split": name,
            "r2": r2_score(y_split, pred),
            "mae": mean_absolute_error(y_split, pred),
            "rmse": np.sqrt(mean_squared_error(y_split, pred))
        }

    scores = {
        "train": _eval("train", X_train, y_train),
        "val": _eval("val", X_val, y_val),
        "test": _eval("test", X_test, y_test),
        "best_params": search.best_params_
    }

    return best_model, scores


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
train_rf_with_search_time_split